In [1]:
import os, time, socket
import numpy as np, pandas as pd, torch
from wasserstein_ica import WassersteinICA
torch.set_num_threads(1)   # leave the CPUs to E1's baselines
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(socket.gethostname(), device)
OUT = "results/rebuttal_contrast_ablation"; os.makedirs(OUT, exist_ok=True)
F_AB = f"{OUT}/ablation.csv"
DIMS = [10, 20, 30]
POOLS = ['Continuous Only', 'Discrete Only', 'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid']
TRIALS = range(10)
N = 10_000
VARIANTS = ['Logcosh pipeline', 'OT-ICA K=1']


gpux2 cuda


In [2]:
# =============================================================================
# Amari performance index
# =============================================================================
def amari_error(W, A):
    """
    W : estimated unmixing (d×d)
    A : true mixing (d×d)
    Returns scalar ∈ [0, 1]; 0 = perfect separation.
    """
    if W is None or np.any(np.isnan(W)):
        return np.nan
    P = np.abs(W @ A)
    n = P.shape[0]
    # Sum each normalised row/column first, THEN subtract 1 once per row/column.
    # (Subtracting 1 inside np.sum would apply it to every element — d× too many.)
    row_terms = np.sum(P / np.max(P, axis=1, keepdims=True), axis=1) - 1.0
    col_terms = np.sum(P / np.max(P, axis=0, keepdims=True), axis=0) - 1.0
    return (np.sum(row_terms) + np.sum(col_terms)) / (2.0 * n)

In [3]:
# =============================================================================
# Data generator (exact port of ablation notebook)
# =============================================================================
def generate_mixture(n_dim, n_samples, config_type, seed=None):
    """
    config_type ∈ {'Continuous Only', 'Discrete Only',
                   'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid'}.
    """
    if seed is not None:
        np.random.seed(seed)

    def gen_laplace():     return np.random.laplace(0, 1/np.sqrt(2), n_samples)
    def gen_bernoulli():   return np.random.choice([-1., 1.], n_samples)
    def gen_uniform():     return np.random.uniform(-np.sqrt(3), np.sqrt(3), n_samples)
    def gen_student_t():   s = np.random.standard_t(3, n_samples); return s / np.std(s)
    def gen_poisson():     s = np.random.poisson(3, n_samples); return (s - s.mean()) / s.std()
    def gen_binomial():    s = np.random.binomial(10, .5, n_samples); return (s - s.mean()) / s.std()
    def gen_chisquare():   s = np.random.chisquare(2, n_samples); return (s - s.mean()) / s.std()
    def gen_exponential(): s = np.random.exponential(1, n_samples); return (s - s.mean()) / s.std()

    pools = {
        'Full Hybrid':             [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        'Continuous Only':         [gen_laplace, gen_uniform, gen_student_t,
                                    gen_chisquare, gen_exponential],
        'Discrete Only':           [gen_bernoulli, gen_poisson, gen_binomial],
        'Strictly Super-Gaussian': [gen_laplace, gen_student_t, gen_chisquare, gen_exponential],
        'Zero Gaussian':           [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        # Pure single-distribution configs (no Gaussian component)
        'Pure Laplace':            [gen_laplace],
        'Pure Uniform':            [gen_uniform],
        'Pure Student-t':          [gen_student_t],
        'Pure Bernoulli':          [gen_bernoulli],
        'Pure Poisson':            [gen_poisson],
        'Pure Binomial':           [gen_binomial],
        'Pure Chi-square':         [gen_chisquare],
        'Pure Exponential':        [gen_exponential],
    }

    active_pool = pools[config_type]
    sources = []
    is_pure = config_type == 'Zero Gaussian' or config_type.startswith('Pure ')
    if not is_pure:
        sources.append(np.random.normal(0, 1, n_samples))
        n_to_gen = n_dim - 1
    else:
        n_to_gen = n_dim

    for _ in range(n_to_gen):
        sources.append(np.random.choice(active_pool)())

    S = np.stack(sources)
    np.random.shuffle(S)

    cond_num = 1000.0
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
    return A @ S, A

In [4]:
_zq, _wq = np.polynomial.hermite_e.hermegauss(200)
C_GAUSS = float(np.sum(_wq * np.log(np.cosh(_zq))) / np.sqrt(2 * np.pi))  # E[log cosh(nu)]

class LogcoshICA(WassersteinICA):
    """Identical OT-ICA pipeline, but the contrast is FastICA's negentropy proxy |E log cosh(y) - E log cosh(nu)|."""
    def wasserstein2_analytical(self, W, cost='l2', dither_sigma=0.0):
        is_1d = W.dim() == 1
        if is_1d: W = W.unsqueeze(0)
        Y = torch.mm(W, self.X_white)
        if dither_sigma > 0:
            Y = Y + torch.randn_like(Y) * dither_sigma
        logcosh = Y.abs() + torch.log1p(torch.exp(-2 * Y.abs())) - np.log(2.0)   # overflow-safe log cosh
        J = torch.abs(logcosh.mean(dim=1) - C_GAUSS)
        return J[0] if is_1d else J


In [5]:
def run_variant(dim, trial, config, variant):
    X, A = generate_mixture(dim, N, config, seed=trial)
    torch.manual_seed(trial)
    cls = LogcoshICA if variant == 'Logcosh pipeline' else WassersteinICA
    K = 1 if variant == 'OT-ICA K=1' else min(dim * 4, 150)
    t0 = time.perf_counter()
    ica = cls(torch.tensor(X, dtype=torch.float32, device=device)); ica.whiten()
    ex = []
    for _ in range(dim):
        prev = torch.stack(ex) if ex else None
        w, _ = ica.optimize_wasserstein2(prev_components=prev, max_iter=200, n_restarts=K, dither_sigma=0.01)
        ex.append(w)
    W = ica.optimize_symmetric(n_components=dim, max_iter=400, lr=0.25, init_w=torch.stack(ex),
                               optimizer='stiefel', dither_sigma=0.01, batch_size=1024)
    err = amari_error(W.cpu().numpy() @ ica.W_white.cpu().numpy(), A)
    return {"Dim": dim, "Config": config, "Trial": trial, "Method": variant, "Amari": err,
            "Time": time.perf_counter() - t0, "Device": device}

done = set()
if os.path.exists(F_AB):
    done = set(map(tuple, pd.read_csv(F_AB)[["Dim", "Config", "Trial", "Method"]].itertuples(index=False)))
for dim in DIMS:
    for config in POOLS:
        for trial in TRIALS:
            for variant in VARIANTS:
                if (dim, config, trial, variant) in done: continue
                row = run_variant(dim, trial, config, variant)
                pd.DataFrame([row]).to_csv(F_AB, mode="a", header=not os.path.exists(F_AB), index=False)
                with open(f"{OUT}/progress.log", "a") as fh:
                    fh.write(f"{time.strftime('%H:%M:%S')} d={dim} {config} trial={trial} {variant}: amari={row['Amari']:.3f} t={row['Time']:.0f}s\n")
print("E2 done")

E2 done
